# Attentional Blink for LLMs — Colab notebook
Tests whether an AB-like phenomenon appears in an LLM, using:
- **conscious** = T2 appears in the tokenized output (binary report)
- **unconscious strength** = joint log-prob of the correct T2 under the forced template

Runtime → Change runtime type → **T4 GPU**. See `../LITERATURE.md` for the design rationale and the honest feasibility caveats.

In [ ]:
!pip -q install "transformers>=4.44" accelerate
# bitsandbytes only needed for 7B 4-bit:
# !pip -q install bitsandbytes

In [ ]:
# Make LLM_Blink importable. If you cloned the repo, this folder already sits next to the notebook.
# In a bare Colab, upload the LLM_Blink/ folder (Files panel) or git clone your repo, then:
import sys, os
sys.path.insert(0, os.getcwd())            # adjust if LLM_Blink/ is elsewhere
from LLM_Blink import load_model, run_sweep, plot_ab, build_trial, TrialConfig
print("LLM_Blink imported OK")

## 1. Load a small instruct model (free-tier friendly)

In [ ]:
model, tok = load_model("Qwen/Qwen2.5-3B-Instruct")  # fp16, fits T4
# alternatives: "Qwen/Qwen2.5-1.5B-Instruct" (fast), "google/gemma-2-2b-it"
# for 7B: load_model("Qwen/Qwen2.5-7B-Instruct", load_in_4bit=True)

## 2. Inspect one trial (sanity check the stimulus)

In [ ]:
tr = build_trial(TrialConfig(lag=2, t1_load="semantic_4", regime="cot", seed=0))
print(tr.user_prefix)
print("\nT2 to detect:", tr.t2_phrase, "| T1 answer:", tr.t1_answer)

## 3. Pilot sweep
2 loads x 6 lags x 2 regimes x n_seeds trials. Start small (`n_seeds=10`), scale up once it looks right.
`do_generation=True` also runs greedy decoding for the binary report measure (slower).

In [ ]:
df = run_sweep(
    model, tok,
    lags=(0, 2, 4, 6, 8, 10),
    loads=("none", "semantic_4"),       # baseline + hardest semantic level
    regimes=("cot", "direct"),          # contrast generation-dynamics vs encoding blink
    n_seeds=10,
    do_generation=True,
)
df.to_csv("ab_results.csv", index=False)
df.head()

## 4. The AB figure
One panel per regime, with both measures stacked. Look for: `T1=semantic_4` **dips** at intermediate lags and **recovers** by the longest ones, while `T1=none` stays **flat** (positional baseline). The CoT/direct contrast separates a generation-dynamics blink from an encoding one. All flat → no blink in this regime (informative null).

In [ ]:
import matplotlib.pyplot as plt
regimes_present = sorted(df["regime"].unique())
has_correct = "report_correct" in df.columns
n_rows = 2 if has_correct else 1
fig, axes = plt.subplots(
    n_rows, len(regimes_present),
    figsize=(6 * len(regimes_present), 4 * n_rows),
    squeeze=False,
)
for col, regime in enumerate(regimes_present):
    plot_ab(df, "t2_mean_logprob", regime=regime, ax=axes[0, col])   # graded 'unconscious'
    if has_correct:
        plot_ab(df, "report_correct", regime=regime, ax=axes[1, col])  # binary 'conscious'
plt.tight_layout(); plt.show()

## 5. Sanity: was the T1 load actually performed?
If `t1_correct` is near 0 for `semantic_4`, the model isn't really paying the load cost — interpret with care.

In [ ]:
if "t1_correct" in df.columns:
    print(df.groupby("t1_load")["t1_correct"].mean(dropna=True))

## Next steps
- Re-run with `regimes=("direct",)` and compare (encoding vs generation-dynamics blink).
- Add a second model family (e.g. gemma-2-2b-it) — single-model effects are often idiosyncratic.
- Titrate toward ~50% report via T2 length / mask / distractor similarity (see ../PROMPTS.md P2).